# LLM Multi-Turn Chat

Builds on `llm_chat_basics.ipynb` by adding conversation memory to the Gemini API. Covers why raw API calls are stateless, how to structure and replay conversation history manually, and a continuous chat loop with per-call error handling that survives individual API failures without ending the session.

## Import Statements

In [ ]:
import os
from dotenv import load_dotenv
import google.genai as genai
from google.genai import errors, types
import requests.exceptions

## Load API Key

In [ ]:
# Searches the custom .env file, reads all the keys value pairs inside it, and then loads them into our computer's temp memory background
# (environment variables) while the notebook is running.
# It works only if the file is present in the same folder as the .ipynb file.
load_dotenv(dotenv_path = "llm_api_variables.env")

# Pull the key out of the memory and assign it to a variable
gemini_Api_Key = os.getenv("GEMINI_API_KEY")
if not gemini_Api_Key:
    print("Loading failed! No such API KEY available")
else:
    print(f"Key loaded successfully! First 5 characters of the key are: {gemini_Api_Key[:5]}")

## Create Client

In [ ]:
client = genai.Client(api_key = gemini_Api_Key)

## Save Conversation History

In [ ]:
conversation_history = []
def save_conversation_history(role, parts):
    conversation_history.append({"role": role, "parts" : parts})
    return conversation_history

## Format the user input and model response

In [ ]:
def format_text(text):
    L = []
    L.append({"text": text})
    return L

## Multi Chat Code

In [ ]:
while True:
    user_input = input()
    print("User typed: ", user_input)
    if(user_input.lower() == "exit"):
        break
    else:
        save_conversation_history("user", format_text(user_input))
        try:
            gemini = client.models.generate_content(
                model = "gemini-3.6-flash",
                contents = conversation_history,
                config = types.GenerateContentConfig(
                    automatic_function_calling = types.AutomaticFunctionCallingConfig(
                        disable = True
                    )
                )
            )
            save_conversation_history("model", format_text(gemini.text))
            print("Gemini response: ", gemini.text)

        except errors.ClientError as e:
            if e.code in (401, 403):
                print("API key error: Your key is invalid or unauthorized.")
            elif e.code == 429:
                print("Rate limit error: Too many requests were sent too quickly.")
            else:
                print(f"API request error ({e.code}): {e.message}")
            print(f"Details: {e}")

        except errors.ServerError as e:
            print("Gemini server error: Google's service is temporarily unavailable.")
            print(f"Details: {e}")

        except (requests.exceptions.ConnectionError, OSError) as e:
            print("Network error: Could not connect to the internet or Google's servers.")
            print(f"Details: {e}")

        except Exception as e:
            print("Unexpected error: Something else went wrong.")
            print(f"Details: {e}")